# COLMAP BA v3 — Sequential Matching + Procrustes Alignment (GPU)

**Runtime**: GPU (T4) — Runtime > Change runtime type

**Changes from v2**:
- Sequential matching (not exhaustive) — correct for ordered keyframes
- Procrustes alignment to fix gauge drift (scale/rotation/translation)
- Downloads aligned poses ready for OpenMVS

**Input**: Upload `scan_data.zip` from Desktop (images/ + sparse/)

In [ ]:
# Cell 1: Install
!pip install pycolmap-cuda12 scipy
import pycolmap
print("version:", pycolmap.__version__, "| has_cuda:", pycolmap.has_cuda)

In [ ]:
# Cell 2: Upload scan_data.zip
from google.colab import files
import os, zipfile
uploaded = files.upload()
os.makedirs('/content/work', exist_ok=True)
with zipfile.ZipFile('scan_data.zip', 'r') as z:
    z.extractall('/content/work')
print('Images:', len(os.listdir('/content/work/images')))
print('Sparse:', os.listdir('/content/work/sparse'))

In [ ]:
# Cell 3: Feature Extraction (GPU)
import pycolmap, os, time
os.chdir('/content/work')
if os.path.exists('database.db'):
    os.remove('database.db')

reader_opts = pycolmap.ImageReaderOptions()
reader_opts.camera_model = 'PINHOLE'
reader_opts.camera_params = '1428.3756,1428.3756,958.9948,725.38055'

t0 = time.time()
pycolmap.extract_features(
    database_path='database.db',
    image_path='images',
    camera_mode=pycolmap.CameraMode.SINGLE,
    reader_options=reader_opts,
    device=pycolmap.Device.cuda
)
print(f'=== Feature extraction: {time.time()-t0:.1f}s ===')

In [ ]:
# Cell 4: Sequential Matching (GPU)
import pycolmap, time
os.chdir('/content/work')

t0 = time.time()
pycolmap.match_sequential(
    database_path='database.db',
    matching_options=pycolmap.SequentialMatchingOptions(
        overlap=15,
        quadratic_overlap=True
    )
)
print(f'=== Sequential matching: {time.time()-t0:.1f}s ===')

# Verify match count
import sqlite3
conn = sqlite3.connect('database.db')
n_pairs = conn.execute('SELECT COUNT(*) FROM matches').fetchone()[0]
n_verified = conn.execute('SELECT COUNT(*) FROM two_view_geometries WHERE rows > 0').fetchone()[0]
conn.close()
print(f'Match pairs: {n_pairs} | Verified: {n_verified}')
print(f'(Exhaustive would be {180*179//2} pairs — sequential should be much fewer)')

In [ ]:
# Cell 5: Triangulate + Bundle Adjustment
import pycolmap, os, shutil, time
os.chdir('/content/work')

rec = pycolmap.Reconstruction('sparse')
print(f'Before triangulation: {rec.summary()}')

for d in ['sparse_triangulated', 'sparse_refined']:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d)

t0 = time.time()
rec = pycolmap.triangulate_points(
    reconstruction=rec,
    database_path='database.db',
    image_path='images',
    output_path='sparse_triangulated',
    refine_intrinsics=False
)
print(f'After triangulation ({time.time()-t0:.1f}s): {rec.summary()}')

t0 = time.time()
ba_options = pycolmap.BundleAdjustmentOptions()
ba_options.refine_focal_length = False
ba_options.refine_principal_point = False
ba_options.refine_extra_params = False
pycolmap.bundle_adjustment(rec, ba_options)
print(f'After BA ({time.time()-t0:.1f}s): {rec.summary()}')

rec.write_text('sparse_refined')
print(f'Saved refined poses to sparse_refined/')

In [ ]:
# Cell 6: Procrustes Alignment (refined → ARKit frame)
import numpy as np
from scipy.spatial.transform import Rotation

def parse_poses(path):
    poses = {}
    with open(path) as f:
        lines = [l.strip() for l in f if not l.startswith('#') and l.strip()]
    for i in range(0, len(lines), 2):
        parts = lines[i].split()
        if len(parts) >= 10 and parts[9].endswith('.jpg'):
            poses[parts[9]] = [float(x) for x in parts[1:8]]
    return poses

def cam_pos(qw, qx, qy, qz, tx, ty, tz):
    R = Rotation.from_quat([qx, qy, qz, qw]).as_matrix()
    return -R.T @ np.array([tx, ty, tz])

def procrustes(src, dst):
    mu_s, mu_d = src.mean(0), dst.mean(0)
    sc, dc = src - mu_s, dst - mu_d
    U, S, Vt = np.linalg.svd(sc.T @ dc)
    d = np.linalg.det(Vt.T @ U.T)
    R = Vt.T @ np.diag([1, 1, d]) @ U.T
    scale = np.trace(R @ sc.T @ dc) / np.trace(sc.T @ sc)
    t = mu_d - scale * R @ mu_s
    return scale, R, t

def apply_sim(qw, qx, qy, qz, tx, ty, tz, s, R_s, t_s):
    R_c = Rotation.from_quat([qx, qy, qz, qw]).as_matrix()
    t_c = np.array([tx, ty, tz])
    R_new = R_c @ R_s.T / s
    t_new = t_c - R_new @ t_s
    U2, _, Vt2 = np.linalg.svd(R_new)
    R_new = U2 @ Vt2
    q = Rotation.from_matrix(R_new).as_quat()  # xyzw
    return q[3], q[0], q[1], q[2], t_new[0], t_new[1], t_new[2]

orig = parse_poses('sparse/images.txt')
refined = parse_poses('sparse_refined/images.txt')
common = sorted(set(orig) & set(refined))
print(f'Original: {len(orig)} | Refined: {len(refined)} | Common: {len(common)}')

src_pts = np.array([cam_pos(*refined[n]) for n in common])
dst_pts = np.array([cam_pos(*orig[n]) for n in common])
scale, R_sim, t_sim = procrustes(src_pts, dst_pts)

aligned = scale * (R_sim @ src_pts.T).T + t_sim
errors = np.linalg.norm(aligned - dst_pts, axis=1)
print(f'Scale: {scale:.4f} | Rotation: {np.degrees(np.arccos(np.clip((np.trace(R_sim)-1)/2, -1, 1))):.2f} deg')
print(f'Residual — Mean: {errors.mean()*100:.2f}cm | Max: {errors.max()*100:.2f}cm')
print(f'Expected: 1-3cm mean = BA is refining well')

# Write aligned poses
import shutil
os.makedirs('sparse_aligned', exist_ok=True)
shutil.copy('sparse_refined/cameras.txt', 'sparse_aligned/')
shutil.copy('sparse_refined/points3D.txt', 'sparse_aligned/')

with open('sparse_refined/images.txt') as fin, open('sparse_aligned/images.txt', 'w') as fout:
    for line in fin:
        if line.startswith('#') or line.strip() == '':
            fout.write(line)
            continue
        parts = line.strip().split()
        if len(parts) >= 10 and parts[9].endswith('.jpg'):
            qw, qx, qy, qz = [float(x) for x in parts[1:5]]
            tx, ty, tz = [float(x) for x in parts[5:8]]
            qw2, qx2, qy2, qz2, tx2, ty2, tz2 = apply_sim(qw, qx, qy, qz, tx, ty, tz, scale, R_sim, t_sim)
            fout.write(f"{parts[0]} {qw2:.17e} {qx2:.17e} {qy2:.17e} {qz2:.17e} "
                       f"{tx2:.17e} {ty2:.17e} {tz2:.17e} {parts[8]} {parts[9]}\n")
        else:
            fout.write(line)

print(f'Aligned poses written to sparse_aligned/')

In [ ]:
# Cell 7: Download aligned poses
import zipfile, os
from google.colab import files
os.chdir('/content/work')
with zipfile.ZipFile('/content/aligned_output.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir('sparse_aligned'):
        zf.write(f'sparse_aligned/{f}', f'sparse_aligned/{f}')
    # Also include raw refined for comparison
    for f in os.listdir('sparse_refined'):
        zf.write(f'sparse_refined/{f}', f'sparse_refined/{f}')
files.download('/content/aligned_output.zip')
print('Download complete — unzip into openmvs_work/ and run TextureMesh')